Week 2 : Prepare corpus

In [ ]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

corpus = [
    "I love this movie it is amazing",
    "This film was terrible and boring",
    "The acting was great and story was good",
    "I did not like the movie"
]

tokenized_corpus = [word_tokenize(sent.lower()) for sent in corpus]


Train Word2Vec

In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

word_vectors = w2v_model.wv
print(word_vectors["movie"].shape)


Pretrained GloVe

In [ ]:
import gensim.downloader as api
glove = api.load("glove-wiki-gigaword-100")
print(glove["movie"])


Dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("imdb", split="train[:5000]")
texts = dataset["text"]
labels = dataset["label"]


Tokenization & Padding

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=200)
y = labels


LSTM Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=10000, output_dim=100, input_length=200),
    LSTM(128),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(X, y, epochs=3, batch_size=64)


Extract vectors

In [ ]:
import numpy as np

words = list(word_vectors.key_to_index.keys())[:50]
vectors = np.array([word_vectors[word] for word in words])


PCA Visualization

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
reduced = pca.fit_transform(vectors)

plt.figure(figsize=(8,6))
for i, word in enumerate(words):
    plt.scatter(reduced[i,0], reduced[i,1])
    plt.annotate(word, (reduced[i,0], reduced[i,1]))
plt.show()
